<a href="https://colab.research.google.com/github/luizalmin-netizen/js-developer-pokedex/blob/main/Controle_de_Transa%C3%A7%C3%B5es_Financeiras_com_POO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Implementando Encapsulamento e Polimorfismo no Sistema Bancário

**Encapsulamento**

Encapsulamento é o princípio de agrupar dados (atributos) e métodos (funções) que operam sobre esses dados em uma única unidade, chamada de classe. Ele também envolve restringir o acesso direto a alguns dos componentes do objeto, protegendo a integridade dos dados e expondo apenas a interface necessária para interação.

No seu sistema bancário, você pode aplicar encapsulamento da seguinte forma:

1.  **Classes de Entidade (e.g., `Conta`, `Cliente`, `Transacao`, `Investimento`):**
    *   **Atributos Privados:** Declare os atributos das suas classes (e.g., `saldo` na `Conta`, `nome` no `Cliente`, `valor` na `Transacao`) como privados (`private` em Java/Python com `__` ou convenção `_`). Isso significa que eles só podem ser acessados ou modificados diretamente pelos métodos da própria classe.
    *   **Métodos Públicos (Getters e Setters):** Forneça métodos públicos (ou `getters` e `setters`) para acessar e modificar esses atributos de forma controlada. Por exemplo, em vez de acessar `conta.saldo` diretamente, você teria `conta.getSaldo()` e `conta.depositar(valor)` ou `conta.sacar(valor)`. Os métodos `depositar` e `sacar` encapsulariam a lógica de validação (e.g., saldo suficiente para saque).

    ```python
    class Conta:
        def __init__(self, numero, titular, saldo_inicial=0):
            self.__numero = numero  # Atributo privado
            self.__titular = titular # Atributo privado
            self.__saldo = saldo_inicial # Atributo privado

        def get_saldo(self):
            return self.__saldo # Método público para acessar o saldo

        def depositar(self, valor):
            if valor > 0:
                self.__saldo += valor
                print(f"Depósito de {valor} realizado. Novo saldo: {self.__saldo}")
            else:
                print("O valor do depósito deve ser positivo.")

        def sacar(self, valor):
            if valor > 0 and self.__saldo >= valor:
                self.__saldo -= valor
                print(f"Saque de {valor} realizado. Novo saldo: {self.__saldo}")
                return True
            else:
                print("Saldo insuficiente ou valor de saque inválido.")
                return False

        # Outros métodos para titular, número, etc.
    ```

2.  **Lógica de Negócio nos Métodos:** Mantenha a lógica de como os dados são manipulados dentro dos métodos da classe. Por exemplo, a lógica para realizar uma transferência PIX deve estar na classe `Conta` ou em um serviço que opera sobre objetos `Conta`.

**Polimorfismo**

Polimorfismo significa "muitas formas". Na POO, refere-se à capacidade de objetos de diferentes classes serem tratados como objetos de uma classe comum, através de uma interface unificada, e de um mesmo método se comportar de maneiras diferentes dependendo do objeto que o invoca.

Você pode implementar polimorfismo no seu sistema bancário de várias maneiras:

1.  **Herança e Sobrescrita de Métodos (Overriding):**
    *   **Classe Base Abstrata:** Crie uma classe base (ou interface/protocolo em algumas linguagens) como `Conta` ou `Transacao` que define métodos comuns.
    *   **Classes Derivadas:** Tenha classes derivadas como `ContaCorrente`, `ContaPoupanca`, `ContaInvestimento` que herdam de `Conta`, ou `PixCredito`, `PixDebito` que herdam de `Transacao`.
    *   **Métodos Sobrescritos:** Cada classe derivada pode sobrescrever o comportamento de um método da classe base. Por exemplo, o método `calcularJuros()` pode ser diferente para `ContaPoupanca` e `ContaInvestimento`.

    ```python
    # Exemplo com Herança e Polimorfismo
    from abc import ABC, abstractmethod

    class ContaBancaria(ABC):
        def __init__(self, numero, titular, saldo_inicial):
            self._numero = numero
            self._titular = titular
            self._saldo = saldo_inicial

        @abstractmethod
        def sacar(self, valor):
            pass

        def depositar(self, valor):
            if valor > 0:
                self._saldo += valor
                print(f"Depósito de {valor} realizado na conta {self._numero}. Saldo atual: {self._saldo}")
            else:
                print("O valor do depósito deve ser positivo.")

        def get_saldo(self):
            return self._saldo

    class ContaCorrente(ContaBancaria):
        def __init__(self, numero, titular, saldo_inicial=0, limite_cheque_especial=500):
            super().__init__(numero, titular, saldo_inicial)
            self.__limite_cheque_especial = limite_cheque_especial

        def sacar(self, valor):
            if valor > 0 and (self._saldo + self.__limite_cheque_especial) >= valor:
                self._saldo -= valor
                print(f"Saque de {valor} realizado na Conta Corrente {self._numero}. Saldo atual: {self._saldo}")
                return True
            else:
                print("Saldo insuficiente (incluindo cheque especial) ou valor de saque inválido.")
                return False

    class ContaPoupanca(ContaBancaria):
        def __init__(self, numero, titular, saldo_inicial=0, taxa_juros=0.01):
            super().__init__(numero, titular, saldo_inicial)
            self.__taxa_juros = taxa_juros

        def sacar(self, valor):
            if valor > 0 and self._saldo >= valor:
                self._saldo -= valor
                print(f"Saque de {valor} realizado na Conta Poupança {self._numero}. Saldo atual: {self._saldo}")
                return True
            else:
                print("Saldo insuficiente ou valor de saque inválido.")
                return False

        def render_juros(self):
            juros = self._saldo * self.__taxa_juros
            self._saldo += juros
            print(f"Juros de {juros:.2f} rendidos. Novo saldo: {self._saldo}")

    # Uso Polimórfico
    contas = [
        ContaCorrente("CC001", "Alice", 1000),
        ContaPoupanca("CP002", "Bob", 5000)
    ]

    for conta in contas:
        print(f"\nConta: {conta._numero}, Titular: {conta._titular}, Saldo: {conta.get_saldo()}")
        conta.sacar(200)
        if isinstance(conta, ContaPoupanca):
            conta.render_juros()
    ```

2.  **Interfaces/Classes Abstratas:** Defina uma interface (ou uma classe abstrata com métodos abstratos) para `Investimento` com métodos como `aplicar()`, `resgatar()`, `calcularRentabilidade()`. Então, `CDB`, `LCI`, `TesouroDireto` poderiam ser classes que implementam essa interface, cada uma com sua própria lógica para esses métodos.

3.  **Transferência PIX:** Uma função de transferência PIX poderia aceitar dois objetos `ContaBancaria` (o remetente e o destinatário) como parâmetros, independentemente de serem `ContaCorrente` ou `ContaPoupanca`, usando os métodos polimórficos `sacar` e `depositar` definidos na classe base.

Ao aplicar esses conceitos, você construirá um sistema mais modular, flexível, fácil de manter e estender.

### Estruturando um Menu Interativo no Console

Para criar um menu interativo, utilizaremos um loop `while` que continuará exibindo as opções até que o usuário decida sair. Dentro do loop, apresentaremos as opções, leremos a escolha do usuário e, com base nessa escolha, chamaremos as funções ou métodos apropriados do nosso sistema bancário.

Vamos implementar um exemplo que permite:

1.  **Criar Contas:** Corrente ou Poupança.
2.  **Depositar:** Em uma conta existente.
3.  **Sacar:** De uma conta existente.
4.  **Ver Saldo:** De uma conta específica.
5.  **Listar Contas:** Mostrar todas as contas criadas.
6.  **Sair:** Encerrar o programa.

Para gerenciar as contas, usaremos um dicionário onde a chave será o número da conta e o valor será o objeto `ContaBancaria` correspondente. Isso simulará uma persistência em memória para o nosso sistema.

**Observação:** O código abaixo reutilizará as definições das classes `ContaBancaria`, `ContaCorrente`, e `ContaPoupanca` que foram apresentadas na explicação de encapsulamento e polimorfismo. Certifique-se de que essas classes estejam definidas no seu ambiente antes de executar o código do menu.

### Implementando o Sistema de Log de Transações

Para registrar todas as transações, criaremos uma classe `Transacao` para encapsular os detalhes de cada evento (depósito, saque, etc.). Este log será armazenado em uma lista e, assim como as contas, terá suas próprias funções de salvamento e carregamento para persistência em um arquivo JSON separado.

In [ ]:
# Este célula agora está vazia pois seu conteúdo foi movido para a célula principal (5d89f08c).

Agora, vamos integrar o sistema de log ao menu interativo e às operações bancárias existentes. Isso envolve:

*   **Importar `Transacao` e o log** no arquivo principal.
*   **Registrar transações** dentro das funções `depositar` e `sacar`.
*   **Adicionar uma opção ao menu** para exibir o histórico de transações.
*   **Chamar as funções de salvar/carregar transações** no início e antes do final do programa.

In [ ]:
from abc import ABC, abstractmethod
import json
from datetime import datetime # Importar datetime para as transações

# Classe para representar uma transação
class Transacao:
    def __init__(self, tipo, numero_conta, valor, data_hora=None):
        self.tipo = tipo
        self.numero_conta = numero_conta
        self.valor = valor
        self.data_hora = data_hora if data_hora else datetime.now().isoformat()

    def __str__(self):
        return f"Tipo: {self.tipo.capitalize()}, Conta: {self.numero_conta}, Valor: R${self.valor:.2f}, Data/Hora: {self.data_hora}"

    def to_dict(self):
        return {
            'tipo': self.tipo,
            'numero_conta': self.numero_conta,
            'valor': self.valor,
            'data_hora': self.data_hora
        }

    @classmethod
    def from_dict(cls, data):
        return cls(
            data['tipo'],
            data['numero_conta'],
            data['valor'],
            data['data_hora']
        )

# Definições das classes (reutilizadas da explicação anterior)
class ContaBancaria(ABC):
    def __init__(self, numero, titular, saldo_inicial):
        self._numero = numero
        self._titular = titular
        self._saldo = saldo_inicial

    @abstractmethod
    def sacar(self, valor):
        pass

    def depositar(self, valor):
        if valor > 0:
            self._saldo += valor
            print(f"Depósito de {valor} realizado na conta {self._numero}. Saldo atual: {self._saldo}")
            transacoes_log.append(Transacao('deposito', self._numero, valor)) # Registrar transação
        else:
            print("O valor do depósito deve ser positivo.")

    def get_saldo(self):
        return self._saldo

    def get_numero(self):
        return self._numero

    def get_titular(self):
        return self._titular

    def __str__(self):
        return f"Número: {self._numero}, Titular: {self._titular}, Saldo: R${self._saldo:.2f}"

    @abstractmethod
    def to_dict(self):
        pass

class ContaCorrente(ContaBancaria):
    def __init__(self, numero, titular, saldo_inicial=0, limite_cheque_especial=500):
        super().__init__(numero, titular, saldo_inicial)
        self.__limite_cheque_especial = limite_cheque_especial

    def sacar(self, valor):
        if valor > 0 and (self._saldo + self.__limite_cheque_especial) >= valor:
            self._saldo -= valor
            print(f"Saque de {valor} realizado na Conta Corrente {self._numero}. Saldo atual: {self._saldo}")
            transacoes_log.append(Transacao('saque', self._numero, valor)) # Registrar transação
            return True
        else:
            print("Saldo insuficiente (incluindo cheque especial) ou valor de saque inválido.")
            return False

    def __str__(self):
        return f"Conta Corrente - {super().__str__()}, Limite Cheque Especial: R${self.__limite_cheque_especial:.2f}"

    def to_dict(self):
        return {
            'tipo': 'corrente',
            'numero': self._numero,
            'titular': self._titular,
            'saldo': self._saldo,
            'limite_cheque_especial': self.__limite_cheque_especial
        }

class ContaPoupanca(ContaBancaria):
    def __init__(self, numero, titular, saldo_inicial=0, taxa_juros=0.01):
        super().__init__(numero, titular, saldo_inicial)
        self.__taxa_juros = taxa_juros

    def sacar(self, valor):
        if valor > 0 and self._saldo >= valor:
            self._saldo -= valor
            print(f"Saque de {valor} realizado na Conta Poupança {self._numero}. Saldo atual: {self._saldo}")
            transacoes_log.append(Transacao('saque', self._numero, valor)) # Registrar transação
            return True
        else:
            print("Saldo insuficiente ou valor de saque inválido.")
            return False

    def render_juros(self):
        juros = self._saldo * self.__taxa_juros
        self._saldo += juros
        print(f"Juros de {juros:.2f} rendidos. Novo saldo: {self._saldo}")

    def __str__(self):
        return f"Conta Poupança - {super().__str__()}, Taxa Juros: {self.__taxa_juros*100:.2f}%"

    def to_dict(self):
        return {
            'tipo': 'poupanca',
            'numero': self._numero,
            'titular': self._titular,
            'saldo': self._saldo,
            'taxa_juros': self.__taxa_juros
        }

# Dicionário para armazenar as contas (simulação de persistência em memória)
contas = {}
ARQUIVO_CONTAS = 'contas.json'

# Lista global para o log de transações
transacoes_log = []
ARQUIVO_TRANSACOES = 'transacoes.json'

def exibir_menu():
    print("\n--- MENU DO SISTEMA BANCÁRIO ---")
    print("1. Criar Nova Conta")
    print("2. Depositar")
    print("3. Sacar")
    print("4. Ver Saldo")
    print("5. Listar Contas")
    print("6. Salvar Contas")
    print("7. Carregar Contas")
    print("8. Ver Histórico de Transações")
    print("9. Sair")
    print("---------------------------------")

def criar_conta():
    print("\n--- CRIAR NOVA CONTA ---")
    tipo_conta = input("Escolha o tipo de conta (cc para Corrente, cp para Poupança): ").lower()
    numero = input("Digite o número da conta: ")
    titular = input("Digite o nome do titular: ")

    if numero in contas:
        print("Erro: Já existe uma conta com este número.")
        return

    while True:
        try:
            saldo_inicial = float(input("Digite o saldo inicial: "))
            if saldo_inicial < 0:
                print("O saldo inicial não pode ser negativo. Tente novamente.")
            else:
                break
        except ValueError:
            print("Saldo inicial inválido. Por favor, digite um número.")

    if tipo_conta == 'cc':
        conta = ContaCorrente(numero, titular, saldo_inicial)
    elif tipo_conta == 'cp':
        conta = ContaPoupanca(numero, titular, saldo_inicial)
    else:
        print("Tipo de conta inválido. Conta não criada.")
        return

    contas[numero] = conta
    print(f"Conta {tipo_conta.upper()} número {numero} criada com sucesso para {titular}!")

def realizar_deposito():
    print("\n--- DEPOSITAR ---")
    numero = input("Digite o número da conta para depósito: ")
    if numero not in contas:
        print("Erro: Conta não encontrada.")
        return

    while True:
        try:
            valor = float(input("Digite o valor a depositar: "))
            if valor <= 0:
                print("O valor do depósito deve ser positivo. Tente novamente.")
            else:
                break
        except ValueError:
            print("Valor inválido. Por favor, digite um número.")
    contas[numero].depositar(valor)

def realizar_saque():
    print("\n--- SACAR ---")
    numero = input("Digite o número da conta para saque: ")
    if numero not in contas:
        print("Erro: Conta não encontrada.")
        return

    while True:
        try:
            valor = float(input("Digite o valor a sacar: "))
            if valor <= 0:
                print("O valor do saque deve ser positivo. Tente novamente.")
            else:
                break
        except ValueError:
            print("Valor inválido. Por favor, digite um número.")
    contas[numero].sacar(valor)

def verificar_saldo():
    print("\n--- VER SALDO ---")
    numero = input("Digite o número da conta para verificar o saldo: ")
    if numero not in contas:
        print("Erro: Conta não encontrada.")
        return

    print(f"Saldo da conta {numero}: R${contas[numero].get_saldo():.2f}")

def listar_contas():
    print("\n--- LISTA DE CONTAS ---")
    if not contas:
        print("Nenhuma conta cadastrada.")
        return
    for numero, conta in contas.items():
        print(conta)

def exibir_historico_transacoes():
    print("\n--- HISTÓRICO DE TRANSAÇÕES ---")
    if not transacoes_log:
        print("Nenhuma transação registrada ainda.")
        return
    for transacao in transacoes_log:
        print(transacao)

def salvar_contas():
    print("\n--- SALVANDO CONTAS ---")
    dados_para_salvar = {num: conta.to_dict() for num, conta in contas.items()}
    try:
        with open(ARQUIVO_CONTAS, 'w', encoding='utf-8') as f:
            json.dump(dados_para_salvar, f, indent=4)
        print(f"Contas salvas com sucesso em '{ARQUIVO_CONTAS}'.")
    except IOError as e:
        print(f"Erro ao salvar contas: {e}")

def carregar_contas():
    print("\n--- CARREGANDO CONTAS ---")
    global contas
    try:
        with open(ARQUIVO_CONTAS, 'r', encoding='utf-8') as f:
            dados_carregados = json.load(f)

        novas_contas = {}
        for num, data in dados_carregados.items():
            if data['tipo'] == 'corrente':
                conta = ContaCorrente(
                    data['numero'],
                    data['titular'],
                    data['saldo'],
                    data['limite_cheque_especial']
                )
            elif data['tipo'] == 'poupanca':
                conta = ContaPoupanca(
                    data['numero'],
                    data['titular'],
                    data['saldo'],
                    data['taxa_juros']
                )
            else:
                print(f"Tipo de conta desconhecido: {data['tipo']}. Ignorando conta {num}.")
                continue
            novas_contas[num] = conta
        contas = novas_contas
        print(f"Contas carregadas com sucesso de '{ARQUIVO_CONTAS}'.")
    except FileNotFoundError:
        print(f"Arquivo '{ARQUIVO_CONTAS}' não encontrado. Iniciando com contas vazias.")
        contas = {}
    except json.JSONDecodeError as e:
        print(f"Erro ao decodificar JSON do arquivo '{ARQUIVO_CONTAS}': {e}")
        contas = {}
    except IOError as e:
        print(f"Erro ao carregar contas: {e}")
        contas = {}

def salvar_transacoes():
    print("\n--- SALVANDO TRANSAÇÕES ---")
    dados_para_salvar = [t.to_dict() for t in transacoes_log]
    try:
        with open(ARQUIVO_TRANSACOES, 'w', encoding='utf-8') as f:
            json.dump(dados_para_salvar, f, indent=4)
        print(f"Transações salvas com sucesso em '{ARQUIVO_TRANSACOES}'.")
    except IOError as e:
        print(f"Erro ao salvar transações: {e}")

def carregar_transacoes():
    print("\n--- CARREGANDO TRANSAÇÕES ---")
    global transacoes_log
    try:
        with open(ARQUIVO_TRANSACOES, 'r', encoding='utf-8') as f:
            dados_carregados = json.load(f)
        transacoes_log = [Transacao.from_dict(data) for data in dados_carregados]
        print(f"Transações carregadas com sucesso de '{ARQUIVO_TRANSACOES}'.")
    except FileNotFoundError:
        print(f"Arquivo '{ARQUIVO_TRANSACOES}' não encontrado. Iniciando com log vazio.")
        transacoes_log = []
    except json.JSONDecodeError as e:
        print(f"Erro ao decodificar JSON do arquivo '{ARQUIVO_TRANSACOES}': {e}")
        transacoes_log = []
    except IOError as e:
        print(f"Erro ao carregar transações: {e}")
        transacoes_log = []


def main():
    # Tenta carregar contas e transações ao iniciar o programa
    carregar_contas()
    carregar_transacoes()

    while True:
        exibir_menu()
        opcao = input("Escolha uma opção: ")

        if opcao == '1':
            criar_conta()
        elif opcao == '2':
            realizar_deposito()
        elif opcao == '3':
            realizar_saque()
        elif opcao == '4':
            verificar_saldo()
        elif opcao == '5':
            listar_contas()
        elif opcao == '6':
            salvar_contas()
        elif opcao == '7':
            carregar_contas()
        elif opcao == '8':
            exibir_historico_transacoes()
        elif opcao == '9':
            salvar_contas() # Salvar antes de sair
            salvar_transacoes() # Salvar transações antes de sair
            print("Saindo do sistema. Até mais!")
            break
        else:
            print("Opção inválida. Por favor, tente novamente.")

if __name__ == "__main__":
    main()


--- CARREGANDO CONTAS ---
Arquivo 'contas.json' não encontrado. Iniciando com contas vazias.

--- MENU DO SISTEMA BANCÁRIO ---
1. Criar Nova Conta
2. Depositar
3. Sacar
4. Ver Saldo
5. Listar Contas
6. Salvar Contas
7. Carregar Contas
8. Sair
---------------------------------
